# 🚀 Qwen2-VL Video Visual Relation Detection (VidVRD)
**Mục tiêu:** Chạy mô hình VLM (`Qwen/Qwen2-VL-2B-Instruct`) trên Google Colab (GPU T4 15GB VRAM) để suy luận quan hệ hành vi và phát hiện đồ vật bị bỏ quên (`[2] abandon [4]`).

### 📋 Quy trình 3 bước:
1. **Cell 1:** Cài đặt các thư viện cần thiết (`transformers`, `qwen-vl-utils`, `accelerate`).
2. **Cell 2:** Nạp mô hình `Qwen2-VL-2B-Instruct` vào GPU T4 (giới hạn `max_pixels` để tiết kiệm VRAM và tăng tốc độ).
3. **Cell 3:** Tải 8 bức ảnh Set-of-Marks, thực hiện suy luận (`temperature=0`, ép ra raw JSON) và kiểm tra kết quả với bộ 26 quan hệ của Hệ thống.

In [ ]:
# ============================================================
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN & KIỂM TRA GPU T4 (TƯƠNG THÍCH QWEN 3.5)
# ============================================================
# Nâng cấp transformers lên bản mới nhất hỗ trợ Qwen 3.5 & MultimodalLM
!pip install -q --upgrade transformers accelerate torchvision qwen-vl-utils

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu: Thời lượng chạy (Runtime) -> Thay đổi loại thời lượng chạy -> Chọn T4 GPU.")

In [ ]:
# ============================================================
# BƯỚC 2: NẠP MÔ HÌNH QWEN 3.5 2B (UNIFIED MULTIMODAL VLM)
# ============================================================
import torch
try:
    from transformers import AutoModelForMultimodalLM
    ModelClass = AutoModelForMultimodalLM
except ImportError:
    from transformers import AutoModelForVision2Seq
    ModelClass = AutoModelForVision2Seq

from transformers import AutoProcessor

# Sử dụng chính thức mô hình Qwen3.5-2B mới nhất từ Alibaba
model_id = "Qwen/Qwen3.5-2B"
print(f"Đang tải mô hình thế hệ mới: {model_id} từ Hugging Face...")

model = ModelClass.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True
)

print(f"✅ Mô hình {model_id} đã nạp thành công vào GPU T4 với kiến trúc Multimodal LM!")

In [ ]:
# ============================================================
# BƯỚC 3: SUY LUẬN VLM TỰ ĐỘNG (CHUẨN HỌC THUẬT VIDVRD + GIẢI THÍCH REASON)
# ============================================================
import os
import json
import glob
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info

# 1. Tự động tìm ảnh đầu vào trong /content/
image_paths = sorted(glob.glob("/content/*.jpg") + glob.glob("/content/frames/*.jpg"))
if not image_paths:
    raise FileNotFoundError("⚠️ Không tìm thấy ảnh .jpg nào trong /content/. Hãy tải 8 ảnh lên!")

print(f"✅ Đã tìm thấy {len(image_paths)} frames ảnh visual prompt.")

# 2. Tự động đọc Prompt & Từ vựng từ file Payload JSON
payload_files = glob.glob("/content/*payload*.json") + glob.glob("/content/*.json")
payload_files = [p for p in payload_files if "ket_qua" not in p]

if payload_files:
    print(f"📄 Tự động tải cấu hình từ payload: {payload_files[0]}")
    with open(payload_files[0], "r", encoding="utf-8") as f:
        payload_data = json.load(f)
    system_prompt = payload_data.get("vlm_system_prompt", "")
    user_prompt = payload_data.get("vlm_user_prompt", "")
    allowed_relations = payload_data.get("allowed_relations_vocabulary_26", [])
    if allowed_relations:
        vocab_str = ", ".join(allowed_relations)
        if vocab_str not in system_prompt:
            system_prompt += f"\n\nSTRICT ALLOWED 26 RELATIONS VOCABULARY:\n[{vocab_str}]"
else:
    print("ℹ️ Dùng prompt mặc định tổng quát (chuẩn 26 relations + reason):")
    allowed_relations = ['bite', 'carry', 'clean', 'cut', 'drive', 'feed', 'get_off', 'get_on', 'grab', 'hit', 'hold', 'hug', 'kick', 'kiss', 'knock', 'lean_on', 'lick', 'lift', 'play(instrument)', 'pull', 'push', 'ride', 'shake_hand_with', 'throw', 'touch', 'wave']
    system_prompt = (
        "You are an advanced Video Visual Relation Detection (VidVRD) AI for surveillance analytics. "
        "You are given a temporal sequence of video frames with numbered visual marks [ID] identifying subjects and objects. "
        "Your task is to detect all active visual relations occurring between the marked entities over time.\n\n"
        "STRICT CONSTRAINTS:\n"
        f"1. You MUST strictly select relation predicates ONLY from these 26 predefined categories: {allowed_relations}.\n"
        "2. SEMANTIC AFFORDANCE & ROLE RULES:\n"
        "   - Inanimate objects (e.g. handbag) cannot be the subject of action verbs like 'hold' or 'carry'. Only persons can hold or carry items.\n"
        "   - If a person merely walks past an entity without physical contact or purposeful interaction, DO NOT predict relations.\n"
        "3. Output format MUST be strictly a valid JSON object matching this schema:\n"
        "{\n"
        '  "temporal_summary": "<brief 1-sentence description of overall interactions and movements across frames>",\n'
        '  "triplets": [\n'
        '    {\n'
        '      "subject": "[ID]",\n'
        '      "relation": "<predicate>",\n'
        '      "object": "[ID]",\n'
        '      "reason": "<brief explanation of why this relation is selected based on visual evidence>"\n'
        '    }\n'
        "  ]\n"
        "}\n"
        "4. DO NOT output any markdown code blocks, explanations, or conversational text. Output ONLY raw JSON."
    )
    user_prompt = (
        f"Analyze the {len(image_paths)} sequential frames of this surveillance clip. "
        "Perform a systematic pair-by-pair check across the full time duration:\n"
        "- Examine all Person-Person interactions across frames.\n"
        "- Examine all Person-Object interactions across frames.\n"
        "Remember: Inanimate objects cannot hold humans, and walking past is not get_on.\n"
        "First write a brief 1-sentence temporal_summary of observed actions, then list all detected relation triplets with a 'reason' for each.\n"
        "Select predicates strictly from the allowed 26 categories. "
        'Respond strictly with the JSON object: {"temporal_summary": "...", "triplets": [{"subject": "[ID]", "relation": "<verb>", "object": "[ID]", "reason": "..."}]}.'
    )

# 3. Chuẩn bị nội dung Video tuần tự
video_file_urls = [f"file://{os.path.abspath(p)}" for p in image_paths]
user_content = [
    {
        "type": "video",
        "video": video_file_urls,
    },
    {"type": "text", "text": user_prompt}
]

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# 4. Xử lý dữ liệu đầu vào và suy luận trên GPU T4
text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text_prompt],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.05
    )

generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
raw_output = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

# 5. Trích xuất và kiểm tra mảng JSON kết quả kèm cột Reason
clean_text = raw_output.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
try:
    parsed_json = json.loads(clean_text)

    if isinstance(parsed_json, dict):
        summary = parsed_json.get("temporal_summary", "")
        if summary:
            print(f"🎬 Quan sát thời gian (Temporal Summary): {summary}\n")
        triplets = parsed_json.get("triplets", [])
    elif isinstance(parsed_json, list):
        triplets = parsed_json
    else:
        triplets = []

    # Lưu kết quả độc lập ra file JSON
    with open("/content/ket_qua_vlm.json", "w", encoding="utf-8") as f:
        json.dump(triplets, f, indent=2, ensure_ascii=False)

    print("=" * 80)
    print("KẾT QUẢ SUY LUẬN VLM (PREDICTED RELATION TRIPLETS + REASONING):")
    print("=" * 80)
    print(json.dumps(triplets, indent=2, ensure_ascii=False))
    print("-" * 80)
    print(f"{'Subject':<10} | {'Relation':<16} | {'Object':<10} | {'Status':<10} | Reason / Explanation")
    print("-" * 80)
    for t in triplets:
        sub = t.get("subject", "")
        rel = t.get("relation", "")
        obj = t.get("object", "")
        reason = t.get("reason", "N/A")
        is_valid = rel in allowed_relations if allowed_relations else True
        status = "[OK]" if is_valid else "[X] Invalid"
        print(f"{sub:<10} | {rel:<16} | {obj:<10} | {status:<10} | {reason}")
    print("=" * 80)
    print(f"✅ Đã lưu kết quả tự động vào: /content/ket_qua_vlm.json ({len(triplets)} triplets)")
except Exception as e:
    print("Raw output từ mô hình:", raw_output)
    print(f"Lỗi phân tích JSON: {e}")